In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [15]:
# 1. Load the raw population data (D)
# Skip the row with X1, X2, Y so that 'default payment next month' becomes the actual header
df = pd.read_csv('credit_card_defaults.csv', skiprows=1)
print(df.head())

   ID  LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  PAY_0  PAY_2  PAY_3  PAY_4  \
0   1      20000    2          2         1   24      2      2     -1     -1   
1   2     120000    2          2         2   26     -1      2      0      0   
2   3      90000    2          2         2   34      0      0      0      0   
3   4      50000    2          2         1   37      0      0      0      0   
4   5      50000    1          2         1   57     -1      0     -1      0   

   ...  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  \
0  ...          0          0          0         0       689         0   
1  ...       3272       3455       3261         0      1000      1000   
2  ...      14331      14948      15549      1518      1500      1000   
3  ...      28314      28959      29547      2000      2019      1200   
4  ...      20940      19146      19131      2000     36681     10000   

   PAY_AMT4  PAY_AMT5  PAY_AMT6  default payment next month  
0         0         0   

In [16]:
# Strip hidden spaces from all column names
df.columns = df.columns.str.strip()

In [18]:
# Separate the parameters (X) from the target (Y)
X_full = df.drop(columns=['default payment next month'])
Y_full = df['default payment next month']

Sorting Bias

In theoretical statistics, we assume our samples are independent and identically distributed (i.i.d.). In reality, raw financial data is often sorted chronologically, alphabetically, or by risk severity. If you simply slice the first 500 rows of your dataset to act as your labeled set, you risk inheriting these hidden sorting biases. If your labeled set is biased, the expected value of your Rectifier is biased, which instantly destroys the {E}_{PPI} = theta mathematical guarantee we proved earlier.

To achieve an unbiased partition in Python, we do not rely on standard slicing (e.g., data[:500]). Instead, we rely on the numpy.random module, which generates permutations based on a Mersenne Twister pseudo-random number generator, effectively mimicking an i.i.d. random draw from the population.

In [19]:
# 2. Mathematically Partition into Labeled (n) and Unlabeled (N)
X_N, X_n, Y_unlabeled, Y_n = train_test_split(
    X_full, Y_full, 
    test_size=500,        # Explicitly slicing exactly n=500 for our labeled set
    random_state=42,      # Locks the pseudo-random generator seed 
    shuffle=True          # The mathematical sledgehammer for bias
)

By enforcing shuffle=True, the train_test_split function utilizes a pseudo-random number generator to mathematically destroy any chronological or sorted relationships (ensuring covariance between adjacent rows is zero). This guarantees that our labeled set $n$ is a true independent and identically distributed (i.i.d.) representation of the population. Setting random_state=42 freezes the random seed, ensuring that as we debug our complex CPPI variance formulas later, our underlying data doesn't shift underneath us.

In [20]:
# 3. Simulate the 'Unlabeled' reality 
# We explicitly delete the massive array of labels to prevent data leaks
del Y_unlabeled

In [ ]:
#check the number of rows in X_N and X_n
print("Shape of X_N (Unlabeled):", X_N.shape)
print("Shape of X_n (Labeled):", X_n.shape)

Shape of X_N (Unlabeled): (29500, 24)
Shape of X_n (Labeled): (500, 24)


In [ ]:
# Check the first 5 index numbers of the labeled features and labels
# train_test_split guarantees that the row alignment is perfectly preserved.
# The row at index i in X_n corresponds exactly to the label at index i in Y_n. 
# The same perfect row-to-row alignment holds true for X_N and Y_unlabeled.
print("X_n index IDs:", X_n.index[:5].tolist())
print("Y_n index IDs:", Y_n.index[:5].tolist())


X_n index IDs: [2308, 22404, 23397, 25058, 2664]
Y_n index IDs: [2308, 22404, 23397, 25058, 2664]


Establishing our first benchmark: the Classical Estimator

Before we can prove that CPPI is mathematically superior, we need to formally measure the limitations of classical statistics on our tiny, 500-account labeled dataset.

In [24]:
import scipy.stats as stats

# 1. Calculate the Classical Estimate (Sample Mean)
theta_classical = np.mean(Y_n)

# 2. Calculate the Standard Error for a binary variable
n = len(Y_n)
variance_Y = theta_classical * (1 - theta_classical)
se_classical = np.sqrt(variance_Y / n)

# 3. Construct the 95% Confidence Interval using the CLT
# 1.96 is the Z-score for 95% coverage under the Normal distribution
z_score = stats.norm.ppf(0.975) 
ci_classical_lower = theta_classical - (z_score * se_classical)
ci_classical_upper = theta_classical + (z_score * se_classical)

print(f"Classical Estimate: {theta_classical:.4f}")
print(f"95% CI: [{ci_classical_lower:.4f}, {ci_classical_upper:.4f}]")
print(f"CI Width: {(ci_classical_upper - ci_classical_lower):.4f}")

Classical Estimate: 0.2440
95% CI: [0.2064, 0.2816]
CI Width: 0.0753


Here is why that 7.5% spread is problematic in practice:

Financial Impact (Scale): If a bank manages a $100 million credit portfolio, the difference between a 20.6% default rate and a 28.1% default rate is $7.5 million in unexpected cash losses. That margin of error is too large for accurate risk reserving.

Signal vs. Noise (Effect Size): In Causal ML, you often want to measure the impact of an intervention. Suppose the bank implements a new risk policy that successfully reduces defaults by 2%. If your baseline confidence interval has an inherent noise spread of 7.5%, that 2% true signal is completely swallowed by the statistical noise. You cannot definitively prove the policy worked because the CI is too wide.

The Naive ML Estimator (The Illusion of Certainty)

To escape the 7.5% noise spread of the Classical Estimator, we turn to the massive scale of our 29,500 unlabeled accounts.

In [27]:
from xgboost import XGBClassifier

# 1. Train the Machine Learning Model strictly on the small labeled set (n=500)
model = XGBClassifier(random_state=42, eval_metric='logloss')
model.fit(X_n, Y_n)

# 2. Generate predictions on the MASSIVE unlabeled set (N=29,500)
# We slice [:, 1] to get the probability of Class 1 (Default)
predictions_N = model.predict_proba(X_N)[:, 1] 

# 3. Calculate the Naive ML Estimate
theta_ML = np.mean(predictions_N)

# 4. Calculate the Standard Error and CI using the massive N
N = len(X_N)
variance_ML = theta_ML * (1 - theta_ML)
se_ML = np.sqrt(variance_ML / N)

ci_ML_lower = theta_ML - (z_score * se_ML)
ci_ML_upper = theta_ML + (z_score * se_ML)

print(f"Naive ML Estimate: {theta_ML:.4f}")
print(f"95% CI: [{ci_ML_lower:.4f}, {ci_ML_upper:.4f}]")
print(f"CI Width: {(ci_ML_upper - ci_ML_lower):.4f}")

Naive ML Estimate: 0.1975
95% CI: [0.1929, 0.2020]
CI Width: 0.0091


The ML model's 19.7% prediction is precise (a 0.91% width), but because it disagrees with our unbiased classical estimate of 24.4%, we know it is systematically underpredicting defaults by approximately 5%. It is highly confident, and mathematically wrong.

The Baseline Rectifier & PPI Estimator

In [28]:
# 1. Generate predictions on the SMALL labeled set (n=500)
predictions_n = model.predict_proba(X_n)[:, 1]

# 2. Calculate the individual errors (Delta) for all 500 accounts
deltas = predictions_n - Y_n

# 3. Calculate the Rectifier (Empirical Bias)
rectifier = np.mean(deltas)

# 4. Calculate the Final PPI Estimate
theta_PPI = theta_ML - rectifier

print(f"Empirical Bias (Rectifier): {rectifier:.4f}")
print(f"Naive ML Estimate: {theta_ML:.4f}")
print(f"Corrected PPI Estimate: {theta_PPI:.4f}")
print(f"Classical Baseline: {theta_classical:.4f}")

Empirical Bias (Rectifier): 0.0000
Naive ML Estimate: 0.1975
Corrected PPI Estimate: 0.1975
Classical Baseline: 0.2440


Becasue of the overfitting bias, we should split the training ang testing data set and try again to get an unbiased rectification term on which we can compare our Cross PPI with 

In [29]:
# 1. Split the 500 labeled accounts exactly in half to restore independence
X_train, X_eval, Y_train, Y_eval = train_test_split(
    X_n, Y_n, test_size=0.5, random_state=42
)

# 2. Train the model ONLY on the 250 training accounts
model_standard = XGBClassifier(random_state=42, eval_metric='logloss')
model_standard.fit(X_train, Y_train)

# 3. Calculate the Clean Rectifier on the 250 evaluation accounts
predictions_eval = model_standard.predict_proba(X_eval)[:, 1]
deltas_standard = predictions_eval - Y_eval
rectifier_standard = np.mean(deltas_standard)

# 4. Generate predictions on the MASSIVE unlabeled set (N=29,500)
predictions_N_standard = model_standard.predict_proba(X_N)[:, 1]
theta_ML_standard = np.mean(predictions_N_standard)

# 5. Calculate Final Standard PPI Estimate
theta_PPI_standard = theta_ML_standard - rectifier_standard

# 6. Calculate Standard PPI Variance (bottlenecked by n_eval = 250)
variance_deltas = np.var(deltas_standard, ddof=1)
se_PPI_standard = np.sqrt(variance_deltas / len(Y_eval))

ci_PPI_lower = theta_PPI_standard - (z_score * se_PPI_standard)
ci_PPI_upper = theta_PPI_standard + (z_score * se_PPI_standard)

print(f"Clean Empirical Bias (Rectifier): {rectifier_standard:.4f}")
print(f"Standard PPI Estimate: {theta_PPI_standard:.4f}")
print(f"Standard PPI 95% CI: [{ci_PPI_lower:.4f}, {ci_PPI_upper:.4f}]")
print(f"Standard PPI CI Width: {(ci_PPI_upper - ci_PPI_lower):.4f}")

Clean Empirical Bias (Rectifier): -0.0285
Standard PPI Estimate: 0.2347
Standard PPI 95% CI: [0.1751, 0.2942]
Standard PPI CI Width: 0.1191


The standard PPI confidence interval expanded to 11.9%, which is mathematically worse than our 7.5% Classical baseline. We sacrificed expensive data, and our reward was a less precise estimate.

This happened because you were hit by two mathematical penalties simultaneously:

The Denominator Penalty: By cutting the sample size in half, the evaluation denominator dropped to $n_{eval} = 250$, aggressively widening the standard error bounds.

The Weak Model Penalty: XGBoost is data-hungry. Training it on a mere 250 rows generated a weak model. A weak model makes volatile predictions, meaning the variance of our errors ($\text{Var}(\Delta)$ in the numerator) actually increased.We just empirically proved why strict data splitting is statistically unacceptable for small audit datasets.

The CPPI Pipeline (Reclaiming the Data)

We will now implement the 5-fold cross-fitting architecture to reclaim the 250 rows we lost in Standard PPI.

In [31]:
from sklearn.model_selection import KFold

# 1. Setup the 5-fold architecture and storage arrays
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Array to hold the clean predictions for all 500 labeled accounts
predictions_n_cppi = np.zeros(len(Y_n)) 

# Matrix to hold 5 sets of predictions for the 29,500 unlabeled accounts
# Shape: (29500 rows, 5 columns)
predictions_N_matrix = np.zeros((len(X_N), 5)) 

fold_index = 0

# 2. Execute the Cross-Fitting Loop
for train_idx, eval_idx in kf.split(X_n):
    # Split the data based on the K-Fold indices
    X_train_k, X_eval_k = X_n.iloc[train_idx], X_n.iloc[eval_idx]
    Y_train_k, Y_eval_k = Y_n.iloc[train_idx], Y_n.iloc[eval_idx]
    
    # Train the fold-specific model (on 400 rows)
    model_k = XGBClassifier(random_state=42, eval_metric='logloss')
    model_k.fit(X_train_k, Y_train_k)
    
    # Generate clean predictions for the 100 hold-out accounts
    predictions_n_cppi[eval_idx] = model_k.predict_proba(X_eval_k)[:, 1]
    
    # Generate predictions for the massive unlabeled set and store in the matrix column
    predictions_N_matrix[:, fold_index] = model_k.predict_proba(X_N)[:, 1]
    
    fold_index += 1

# 3. Assemble the CPPI Engines
# Calculate the Ensemble ML Average (average across the 5 models, then across all N)
ensemble_predictions_N = np.mean(predictions_N_matrix, axis=1)
theta_ML_cppi = np.mean(ensemble_predictions_N)

# Calculate the Pooled Rectifier using the clean out-of-sample predictions
deltas_cppi = predictions_n_cppi - Y_n
rectifier_cppi = np.mean(deltas_cppi)

# Calculate the Final CPPI Estimate
theta_CPPI = theta_ML_cppi - rectifier_cppi

print(f"CPPI Empirical Bias (Rectifier): {rectifier_cppi:.4f}")
print(f"CPPI Ensemble ML Average: {theta_ML_cppi:.4f}")
print(f"Final CPPI Estimate: {theta_CPPI:.4f}")

CPPI Empirical Bias (Rectifier): -0.0422
CPPI Ensemble ML Average: 0.1920
Final CPPI Estimate: 0.2342


By writing the predictions directly into the predictions_n_cppi array at the specific eval_idx locations, we programmatically guarantee that each account's error ($\Delta_i$) is calculated using a model that never saw its actual label ($Y_i$). We successfully reclaimed our denominator ($n=500$) without triggering the overfitting trap that caused our Rectifier to flatline at 0.0000 earlier.

The shift from 2.8% to 4.2% means your measurement of the bias became mathematically sharper, not that the model got worse.

The Rectifier Shift: In Standard PPI, the -2.8% bias was measured on just 250 rows. It was a noisy, high-variance snapshot of the model's flaws. By using CPPI, you measured the bias across all 500 rows. The -4.2% is a much more accurate, empirically stable representation of the true systematic underprediction.

The ML Average Stability: The similarity between the Naive ML (19.75%) and the Ensemble ML (19.20%) proves the model is highly stable, which is exactly the property we want to exploit.

Machine learning algorithms are fundamentally biased, especially on imbalanced financial data. Because non-defaults heavily outnumber defaults, XGBoost minimizes its overall loss by leaning safely toward 0 (no default). This causes it to systematically underpredict the true default rate.

The core philosophy of PPI is that we do not need the ML model to be unbiased. We only need it to be stable (low variance). If the ensemble consistently outputs ~19% across 29,500 accounts, we have achieved near-zero variance. We then use the rigorous mathematical Rectifier to physically drag that highly stable 19% prediction up to the unbiased reality of 23.4%.

We now have a mathematically unbiased center point ($\hat{\theta}_{CPPI} = 0.2342$) that correctly mirrors your Classical baseline ($0.2440$), but is stabilized by the massive unlabeled dataset.

The CPPI Variance and Confidence Interval

To finalize our estimator, we must translate the corrected variance mathematics into code. We will re-run the exact same KFold splits to isolate the prediction errors ($\Delta_i$) we calculated in the previous step, calculate the variance strictly inside those isolated boundaries, and then average them.

By refusing to calculate the variance across the entire 500-item array at once, we force the math to ignore the correlated predictions between Model 1 and Model 2, surgically removing the $+ 2\text{Cov}(A,B)$ trap.

In [32]:
# 1. Re-instantiate the exact same KFold generator to guarantee matching splits
kf = KFold(n_splits=5, shuffle=True, random_state=42)

fold_variances = []

# 2. Execute the Isolated Variance Loop
for train_idx, eval_idx in kf.split(X_n):
    # Isolate the errors (deltas) strictly by their integer positions
    # We use .iloc because deltas_cppi inherited the pandas index from Y_n
    fold_deltas = deltas_cppi.iloc[eval_idx]
    
    # Calculate the unbiased sample variance (ddof=1) strictly inside this fold
    fold_var = np.var(fold_deltas, ddof=1)
    fold_variances.append(fold_var)

# 3. Average the 5 isolated variances to get the corrected CPPI variance
sigma2_cppi = np.mean(fold_variances)

# 4. Calculate the Standard Error using the FULL labeled sample size (n=500)
n_full = len(Y_n)
se_CPPI = np.sqrt(sigma2_cppi / n_full)

# 5. Construct the Final 95% Confidence Interval
ci_CPPI_lower = theta_CPPI - (z_score * se_CPPI)
ci_CPPI_upper = theta_CPPI + (z_score * se_CPPI)

print(f"Classical CI Width: {(ci_classical_upper - ci_classical_lower):.4f}")
print(f"Standard PPI CI Width (n=250): {(ci_PPI_upper - ci_PPI_lower):.4f}")
print(f"CPPI 95% CI: [{ci_CPPI_lower:.4f}, {ci_CPPI_upper:.4f}]")
print(f"Final CPPI CI Width (n=500): {(ci_CPPI_upper - ci_CPPI_lower):.4f}")

Classical CI Width: 0.0753
Standard PPI CI Width (n=250): 0.1191
CPPI 95% CI: [0.1935, 0.2749]
Final CPPI CI Width (n=500): 0.0815


Look closely at the numbers you just printed. The CPPI width (8.15%) is strictly larger than the Classical width (7.53%). After all that advanced cross-fitting architecture, our cutting-edge causal ML estimator just lost to a basic 19th-century sample mean.

This is a vital lesson in empirical statistics: The Machine Learning Penalty.

Earlier we mathematically assumed that the variance of the model's errors would be fundamentally smaller than the variance of the raw data:$$\sigma^2_\Delta < \text{Var}(Y)$$The empirical output proves the exact opposite just happened. Because the CPPI interval is wider, we know mathematically that the variance of your model's prediction errors is actually larger than the raw variance of the 0s and 1s.Why did this happen? You trained an XGBoost classifier—a highly complex, non-linear, data-hungry tree algorithm—on a mere 400 rows of imbalanced financial data using default parameters. The model overfit the training data and made erratic, noisy probability predictions on the hold-out folds.When a machine learning model is this weak, its errors ($\Delta_i$) swing wildly.

<span style="color: White; font-weight: bold;">PPI does not guarantee a narrower interval. It guarantees a valid interval. If our ML model is worse than guessing the mean, PPI expands the interval to protect us from false certainty.</span>
